<a href="https://colab.research.google.com/github/MetalHeadv3/Inzyrnieria-Big-Data/blob/main/Lab_6_zad_2_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
#zadanie 2
import random
from tqdm import tqdm

def build_dataset(filename, n_rows=100, chunk_size=100000):
    rows = []
    rows.append(header)
    mu = (salary['max'] + salary['min']) / 2
    sigma = 1000

    with open(filename, 'w', encoding='utf-8') as filehandler:

        for id in tqdm(range(1, n_rows + 1), total=n_rows, desc="Building dataset..."):
            row = [
                f'{id}',
                f'{random.choice(firstnames)}',
                f'{random.choice(lastnames)}',
                f"{random.randint(age['min'], age['max'])}",
                f"{round(float(random.normalvariate(mu=mu, sigma=sigma)), 2)}"
            ]
            rows.append(row)
            if id % chunk_size == 0:
                filehandler.writelines([f"{','.join(row)}\n" for row in rows])
                rows = []

In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("EmployeeSchemaInference").getOrCreate()

employee_df = spark.read.csv("employee.csv", header=True, inferSchema=True)

employee_df.printSchema()
employee_df.show(5)


root
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- salary: integer (nullable = true)
 |-- department: string (nullable = true)

+--------------------+---+------+----------+
|                name|age|salary|department|
+--------------------+---+------+----------+
|       Anna Kowalska| 28|  7200|        HR|
|           Jan Nowak| 35| 10500|        IT|
|     Maria Zielińska| 42|  9800|   Finance|
|   Tomasz Wiśniewski| 55| 12000|     Sales|
|Katarzyna Lewando...| 63|  8700| Marketing|
+--------------------+---+------+----------+
only showing top 5 rows



In [5]:
#zadanie3
import time

start = time.time()
count = employee_df.filter(employee_df["salary"] > 10000).count()
elapsed = time.time() - start

print("Liczba pracowników z pensją > 10000:", count)
print("Czas wykonania:", round(elapsed, 3), "sekundy")


Liczba pracowników z pensją > 10000: 3
Czas wykonania: 1.191 sekundy


In [6]:
#zadanie 4
from pyspark.ml.feature import Bucketizer

splits = [10, 20, 30, 40, 50, 60, 70]

bucketizer = Bucketizer(
    splits=splits,
    inputCol="age",
    outputCol="age_bucket"
)


In [7]:
bucketed_df = bucketizer.transform(employee_df)


In [8]:
bucketed_df.select("age", "age_bucket").show(20)


+---+----------+
|age|age_bucket|
+---+----------+
| 28|       1.0|
| 35|       2.0|
| 42|       3.0|
| 55|       4.0|
| 63|       5.0|
| 31|       2.0|
| 47|       3.0|
| 29|       1.0|
| 38|       2.0|
| 60|       5.0|
+---+----------+



In [9]:
bucketed_df.groupBy("age_bucket").count().orderBy("age_bucket").show()


+----------+-----+
|age_bucket|count|
+----------+-----+
|       1.0|    2|
|       2.0|    3|
|       3.0|    2|
|       4.0|    1|
|       5.0|    2|
+----------+-----+

